In [127]:
import copy
import os
import argparse
from erasure.utils.logger import GLogger
import torch
import numpy as np
import random
from erasure.utils.config.local_ctx import Local
from erasure.utils.config.global_ctx import Global, bcolors 
from erasure.core.factory_base import ConfigurableFactory
from erasure.data.datasets.DatasetManager import DatasetManager

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [128]:
config_file = os.path.join("configs","LinkAttack", "Cora_edge.jsonc")

In [129]:
global_ctx = Global(config_file)
global_ctx.factory = ConfigurableFactory(global_ctx)

#Create Dataset
data_manager = global_ctx.factory.get_object( Local( global_ctx.config.data ))
global_ctx.dataset = data_manager

@,-1168975244 | INFO | 996826 - Creating Global Context for: configs/LinkAttack/Cora_edge.jsonc
����#,-1168975189 | INFO | 996826 - Setting seeds to: 0
!,-1168975175 | INFO | 996826 - REMOVAL TYPE SET AS edge
,-1168975174 | INFO | 996826 - Caching System: False.
���$,-1168975159 | INFO | 996826 - Instantiating: torch_geometric.datasets.Planetoid
`g�3#,-1168974987 | INFO | 996826 - Created Configurable: erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource
�s #,-1168974986 | INFO | 996826 - {'class': 'erasure.data.data_sources.TorchGeometricDataSource.TorchGeometricDataSource', 'parameters': {'datasource': {'class': 'torch_geometric.datasets.Planetoid', 'parameters': {'root': 'resources/data', 'name': 'Cora'}, 'preprocess': []}}}
���$,-1168974959 | INFO | 996826 - Instantiating: erasure.data.datasets.DataSplitterGraph.DataSplitterPercentage
���$,-1168974889 | INFO | 996826 - ['all', 'all_shuffled', '-']
���$,-1168974889 | INFO | 996826 - Instantiating

In [130]:
#Create Predictor
current = Local(global_ctx.config.predictor)
current.dataset = data_manager
predictor = global_ctx.factory.get_object(current)
global_ctx.predictor = predictor
global_ctx.logger.info('Global Predictor: ' + str(predictor))

���$,-1168974780 | INFO | 996826 - Instantiating: erasure.model.graphs.SGC_CGU.SGC_CGU
���$,-1168974778 | INFO | 996826 - Instantiating: torch.optim.Adam
���$,-1168974753 | INFO | 996826 - Instantiating: torch.nn.CrossEntropyLoss


graph has edges  torch.Size([2, 10556])
��t,-1168974710 | INFO | 996826 - epoch = 0 ---> loss = 1.9651	 accuracy = 0.0815
graph has edges  torch.Size([2, 10556])
��t,-1168974664 | INFO | 996826 - epoch = 1 ---> loss = 3.4968	 accuracy = 0.4000
graph has edges  torch.Size([2, 10556])
��t,-1168974607 | INFO | 996826 - epoch = 2 ---> loss = 1.9840	 accuracy = 0.6215
graph has edges  torch.Size([2, 10556])
��t,-1168974546 | INFO | 996826 - epoch = 3 ---> loss = 1.2016	 accuracy = 0.7277
graph has edges  torch.Size([2, 10556])
��t,-1168974496 | INFO | 996826 - epoch = 4 ---> loss = 0.8243	 accuracy = 0.8159
graph has edges  torch.Size([2, 10556])
��t,-1168974455 | INFO | 996826 - epoch = 5 ---> loss = 0.7296	 accuracy = 0.8518
graph has edges  torch.Size([2, 10556])
��t,-1168974407 | INFO | 996826 - epoch = 6 ---> loss = 0.7229	 accuracy = 0.8677
graph has edges  torch.Size([2, 10556])
��t,-1168974364 | INFO | 996826 - epoch = 7 ---> loss = 0.7549	 accuracy = 0.8682
graph has edges  torch.S

In [131]:
data_manager.partitions['all'][0][0]

Data(x=[2708, 1433], edge_index=[2, 10556])

In [132]:
data_manager.partitions['all'][0][1]

tensor([3, 4, 4,  ..., 3, 3, 3])

In [133]:
import torch
print(torch.version.cuda)

12.6


In [134]:
print(predictor.optimizer)

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.5
    lr: 0.24999999999999983
    maximize: False
    weight_decay: 0
)


In [135]:
import torch
print(torch.__version__)
print(torch.version.cuda)


2.7.0+cu126
12.6


In [136]:
#Create unlearners 
unlearners = []
unlearners_cfg = global_ctx.config.unlearners
for un in unlearners_cfg:
    current = Local(un)
    current.dataset = data_manager
    current.predictor = copy.deepcopy(predictor)
    unlearners.append( global_ctx.factory.get_object(current) )

`g�3#,-1168969633 | INFO | 996826 - Created Configurable: erasure.unlearners.certified_graph_unlearners.CGU.CGU_edge
graph tensor([[   0,    0,    1,  ..., 2707, 2707, 2707],
        [1862, 2582,    2,  ...,  598, 1473, 2706]])
`g�3#,-1168969533 | INFO | 996826 - Created Configurable: erasure.unlearners.GoldModel.GoldModelGraph
`g�3#,-1168969505 | INFO | 996826 - Created Configurable: erasure.unlearners.composite.Identity


In [137]:
data_manager.partitions['all'][0][0]

Data(x=[2708, 1433], edge_index=[2, 10556])

In [138]:
data_manager.partitions['all'][0][0].edge_index

tensor([[   0,    0,    0,  ..., 2707, 2707, 2707],
        [ 633, 1862, 2582,  ...,  598, 1473, 2706]])

In [139]:
data_manager.partitions['all'].revise_graph_edges([(0,1),(0,2)])[0][0]

Data(x=[2708, 1433], edge_index=[2, 0])

In [143]:
print(len( data_manager.partitions['forget']) )

1


In [140]:
predictor.model.classifier.weight

Parameter containing:
tensor([[-4.9884, -1.7682,  1.6619,  ...,  2.9387, -0.6055, -5.6886],
        [-3.2379,  1.3747,  1.0405,  ...,  4.6833,  2.6435,  1.8831],
        [-1.6624, -0.4407, -1.0346,  ..., -1.9783, -3.3855, -4.0264],
        ...,
        [ 2.2159, -3.5841,  1.3169,  ...,  8.4479,  5.0861,  3.7126],
        [ 2.5377,  1.2119,  0.8642,  ...,  0.4602,  1.3809, -3.0753],
        [ 3.6852,  0.9635,  0.6485,  ..., -2.4656,  3.1807, -3.2083]],
       requires_grad=True)

In [144]:
#Evaluator
current = Local(global_ctx.config.evaluator)
current.unlearners = unlearners
evaluator = global_ctx.factory.get_object(current)

# Evaluations
for unlearner in unlearners:
    global_ctx.logger.info(f'''{bcolors.OKGREEN}####\t\t Evaluating Unlearner {unlearner.__class__.__name__} \t\t####{bcolors.ENDC}''')
    evaluator.evaluate(unlearner,predictor)

`g�3#,-1168709133 | INFO | 996826 - Created Configurable: erasure.evaluations.running.RunTime
�� #,-1168709127 | INFO | 996826 - Function: <function accuracy_score at 0x7f2322471280>
`g�3#,-1168709126 | INFO | 996826 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
Po"#,-1168709125 | INFO | 996826 - Function: <function accuracy_score at 0x7f2322471280>
`g�3#,-1168709124 | INFO | 996826 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
��"#,-1168709123 | INFO | 996826 - Function: <function accuracy_score at 0x7f2322471280>
`g�3#,-1168709123 | INFO | 996826 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
@"�3#,-1168709122 | INFO | 996826 - Function: <function accuracy_score at 0x7f2322471280>
`g�3#,-1168709121 | INFO | 996826 - Created Configurable: erasure.evaluations.measures.TorchSKLearnGraph
`g�3#,-1168709121 | INFO | 996826 - Created Configurable: erasure.evaluations.measures.SaveValues
`g�3#,-1168709120 